# Lab 4: Iceberg + Spark Optimizations - Solution

This notebook contains the complete solution for Lab 4. Use this to verify your implementation or if you get stuck.

## Step 1: File Compaction Strategies - Bin-Packing

In [ ]:
// Create table with many small files
spark.sql("""
  CREATE TABLE iceberg.default.compaction_test (
    id INT,
    category STRING,
    value DOUBLE,
    timestamp TIMESTAMP
  ) USING iceberg
  PARTITIONED BY (category, days(timestamp))
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert data in small batches
for (batch <- 1 to 20) {
  spark.sql(s"""
    INSERT INTO iceberg.default.compaction_test VALUES
    (${batch * 10 + 1}, 'A', 100.0 * batch, TIMESTAMP '2024-01-01 ${batch % 24}:00:00'),
    (${batch * 10 + 2}, 'A', 200.0 * batch, TIMESTAMP '2024-01-01 ${batch % 24}:05:00'),
    (${batch * 10 + 3}, 'B', 150.0 * batch, TIMESTAMP '2024-01-01 ${batch % 24}:10:00')
  """)
}

In [ ]:
// Check file count before compaction
val filesBefore = spark.sql("""
  SELECT COUNT(*) as file_count
  FROM iceberg.default.compaction_test.files
""").collect().head.getLong(0)
println(s"File count before compaction: $filesBefore")

In [ ]:
// Perform bin-packing compaction
spark.sql("""
  CALL iceberg.system.rewrite_data_files(
    'iceberg.default.compaction_test',
    map(
      'min-input-files', '5',
      'target-size-bytes', str(256 * 1024 * 1024)
    )
  )
""")

In [ ]:
// Check file count after compaction
val filesAfter = spark.sql("""
  SELECT COUNT(*) as file_count
  FROM iceberg.default.compaction_test.files
""").collect().head.getLong(0)
println(s"File count after compaction: $filesAfter")

assert(filesAfter < filesBefore, "Compaction should reduce file count")
println("✅ Bin-packing compaction successful!")

## Step 2: Snapshot Management

In [ ]:
// Create table and generate multiple snapshots
spark.sql("""
  CREATE TABLE iceberg.default.snapshot_test (
    id INT,
    value DOUBLE,
    timestamp TIMESTAMP
  ) USING iceberg
  PARTITIONED BY (days(timestamp))
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Generate multiple snapshots through updates
for (i <- 1 to 5) {
  spark.sql(s"""
    INSERT INTO iceberg.default.snapshot_test VALUES
    ($i, 100.0 * $i, TIMESTAMP '2024-01-01 ${i}:00:00')
  """)
  
  spark.sql(s"""
    UPDATE iceberg.default.snapshot_test
    SET value = value * 1.1
    WHERE id = $i
  """)
}

In [ ]:
// List all snapshots
spark.sql("""
  SELECT 
    snapshot_id,
    committed_at,
    summary['operation'] as operation,
    summary['added-files-size'] as added_files_size
  FROM iceberg.default.snapshot_test.snapshots
  ORDER BY committed_at DESC
""").show()

In [ ]:
// Expire old snapshots (keep only recent ones)
spark.sql("""
  CALL iceberg.system.expire_snapshots(
    'iceberg.default.snapshot_test',
    map(
      'retain-last', '3'
    )
  )
""")

In [ ]:
// Verify snapshot expiration
val snapshotCount = spark.sql("""
  SELECT COUNT(*) as snapshot_count
  FROM iceberg.default.snapshot_test.snapshots
""").collect().head.getLong(0)
println(s"Snapshot count after expiration: $snapshotCount")
assert(snapshotCount <= 3, "Should retain at most 3 snapshots")
println("✅ Snapshot management successful!")

## Step 3: Spark Query Planning Optimization

In [ ]:
// Configure Spark for optimal Iceberg query planning
spark.conf.set("spark.sql.iceberg.planning.enabled", "true")
spark.conf.set("spark.sql.iceberg.planning.mode", "distributed")
spark.conf.set("spark.sql.iceberg.pushdown.enabled", "true")

In [ ]:
// Create table for query planning tests
spark.sql("""
  CREATE TABLE iceberg.default.query_planning_test (
    user_id INT,
    product_id INT,
    purchase_date DATE,
    amount DECIMAL(10,2),
    category STRING,
    region STRING
  ) USING iceberg
  PARTITIONED BY (region, years(purchase_date))
  ORDER BY user_id
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert test data
spark.sql("""
  INSERT INTO iceberg.default.query_planning_test VALUES
    (1, 101, DATE '2024-01-01', 100.50, 'electronics', 'west'),
    (2, 102, DATE '2024-01-02', 200.75, 'electronics', 'east'),
    (3, 103, DATE '2024-01-03', 150.25, 'clothing', 'west'),
    (4, 104, DATE '2024-01-04', 300.00, 'electronics', 'east'),
    (5, 105, DATE '2024-01-05', 175.50, 'clothing', 'west')
""")

In [ ]:
// Test query with explain plan
spark.sql("""
  EXPLAIN EXTENDED
  SELECT user_id, SUM(amount) as total_spent
  FROM iceberg.default.query_planning_test
  WHERE region = 'west'
  AND purchase_date >= DATE '2024-01-01'
  GROUP BY user_id
""").show(truncate=false)

In [ ]:
// Run the actual query
spark.sql("""
  SELECT user_id, SUM(amount) as total_spent
  FROM iceberg.default.query_planning_test
  WHERE region = 'west'
  AND purchase_date >= DATE '2024-01-01'
  GROUP BY user_id
""").show()
println("✅ Query planning optimization working!")

## ✅ Lab Completion Checklist

- [x] File compaction strategies implemented and tested
- [x] Snapshot management successfully removes old snapshots
- [x] Spark query planning optimization shows partition pruning

## 🎓 Key Concepts Learned

1. **File Compaction**: Bin-packing strategies for file optimization
2. **Snapshot Management**: Expiring old snapshots to maintain performance
3. **Query Planning**: Spark optimization for Iceberg queries
4. **Performance Monitoring**: Metrics and statistics for optimization